In [0]:
# ==========================================
# WIDGETS - Configuração do Secrets Scope
# ==========================================
dbutils.widgets.text(
    name         = "scope_name",
    defaultValue = "northwind",
    label        = "Secrets Scope Name"
)

SCOPE_NAME = dbutils.widgets.get("scope_name").strip()

print(f"▶ Scope: {SCOPE_NAME}")

In [0]:
import requests
import json
import getpass

# Lê host e token do ambiente Databricks automaticamente
DATABRICKS_HOST  = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

HEADERS = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type":  "application/json"
}

print(f"Host: {DATABRICKS_HOST}")
print("Token carregado automaticamente.")

In [0]:
def create_secrets_scope(scope_name):
    """
    Cria um Secrets Scope no Databricks.
    Se já existir, retorna informação sem erro.
    """
    # Verifica se já existe
    res = requests.get(
        f"{DATABRICKS_HOST}/api/2.0/secrets/scopes/list",
        headers=HEADERS
    )
    res.raise_for_status()
    
    existing_scopes = [s['name'] for s in res.json().get('scopes', [])]
    
    if scope_name in existing_scopes:
        print(f"Secrets Scope '{scope_name}' já existe.")
        return
    
    # Cria novo scope
    res = requests.post(
        f"{DATABRICKS_HOST}/api/2.0/secrets/scopes/create",
        headers=HEADERS,
        json={
            "scope": scope_name,
            "initial_manage_principal": "users"  # todos os users podem gerenciar
        }
    )
    
    if res.ok:
        print(f"✅ Secrets Scope '{scope_name}' criado com sucesso!")
    else:
        print(f"Erro ao criar scope: {res.status_code}")
        print(res.json())
        res.raise_for_status()

create_secrets_scope(SCOPE_NAME)

In [0]:
def put_secret(scope, key, value):
    """Adiciona ou atualiza um secret no scope."""
    res = requests.post(
        f"{DATABRICKS_HOST}/api/2.0/secrets/put",
        headers=HEADERS,
        json={
            "scope": scope,
            "key":   key,
            "string_value": value
        }
    )
    
    if res.ok:
        print(f"  ✓ {key}")
    else:
        print(f"  ✗ {key}: {res.json()}")


print("\nInsira o API token do Kaggle:")
print("(os valores serão armazenados de forma segura no Secrets Scope)\n")

API_TOKEN     = input("API_TOKEN: ").strip()
KAGGLE_USERNAME     = input("KAGGLE_USERNAME: ").strip()
print("\nGravando secrets ...")
put_secret(SCOPE_NAME, "API_TOKEN", API_TOKEN)
put_secret(SCOPE_NAME, "KAGGLE_USERNAME", KAGGLE_USERNAME)
print("\n✅ Secrets configurados!")

In [0]:
# Lista todos os secrets no scope (não mostra os valores, só as keys)
res = requests.get(
    f"{DATABRICKS_HOST}/api/2.0/secrets/list",
    headers=HEADERS,
    params={"scope": SCOPE_NAME}
)
res.raise_for_status()

secrets = res.json().get('secrets', [])

print("=" * 60)
print(f"SECRETS CONFIGURADOS NO SCOPE '{SCOPE_NAME}'")
print("=" * 60)
print("\nKeys disponíveis (valores são confidenciais e não são exibidos):\n")

for secret in secrets:
    print(f"  • {secret['key']}")

print(f"\nTotal: {len(secrets)} secrets")
print("\n✅ Configuração completa!")
print("\nOs notebooks de ingestão agora lerão automaticamente do Secrets Scope.")
print("Você pode remover o arquivo .env do workspace se desejar.")

In [0]:
# Teste de leitura dos secrets (mostra apenas que consegue ler, não exibe os valores)
print("\nTestando leitura dos secrets...\n")

test_keys = [
    "API_TOKEN", "KAGGLE_USERNAME"
]

for key in test_keys:
    try:
        value = dbutils.secrets.get(scope=SCOPE_NAME, key=key)
        # Não exibe o valor, só confirma que conseguiu ler
        masked = "*" * min(len(value), 8) if value else "(vazio)"
        print(f"  ✓ {key:20} → {masked}")
    except Exception as e:
        print(f"  ✗ {key:20} → Erro: {str(e)}")

print("\n✅ Teste concluído!")